# Benchmark Model Machine Learning & Studi Ablasi Fitur Lintas Domain
## Evaluasi Klasifikasi Bottleneck 5G O-RAN Multi-Slice pada Non-RT RIC rApp
**Dataset:** `data/telemetry_dataset.csv` (4.500 sampel, 40 kolom, 8 skenario eksperimen)  
**Lingkup Metodologi:** Partisi temporal dalam-run bebas bocoran (*intra-run temporal split*), perataan titik evaluasi sekuensial GRU dan tabular, metrik per-kelas lengkap, AUPRC One-vs-Rest, dan pengukuran latensi inferensi berulang.


## Tahap 1: Pemuatan Data, Audit Integritas & Formulasi Target
- **Tujuan:** Memuat dataset telemetri, mencatat hash integritas file dan dependensi sistem, mengaudit missing values awal, dan memetakan label domain ke dalam 4 kelas diskrit.
- **Penanganan Nilai Awal:** Nilai `NaN` pada metrik NWDAF di awal run (sebelum polling HTTP pertama selesai) dikeluarkan secara eksplisit dan dilaporkan jumlahnya, sehingga seluruh set fitur dievaluasi pada sampel pengujian yang 100% konsisten.
- **Target:** Kelas ground-truth: `0: BASELINE`, `1: RAN`, `2: CORE`, `3: JOINT`.


In [ ]:
# 1. Import library analisis dan konfigurasi sistem
import os
import sys
import time
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_fscore_support, f1_score, precision_score, recall_score,
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    precision_recall_curve, auc, average_precision_score
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn

# Konfigurasi Reproduksibilitas & Visualisasi
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 9.5
plt.rcParams['figure.titlesize'] = 11.5
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 10.5
plt.rcParams['figure.dpi'] = 120

# 2. Verifikasi Hash & Pemuatan Dataset
DATASET_PATH = os.path.join('..', 'data', 'telemetry_dataset.csv')
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = os.path.join('data', 'telemetry_dataset.csv')
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = os.path.join('data', 'telemetry_dataset_10min_complete.csv')

with open(DATASET_PATH, 'rb') as f:
    dataset_sha256 = hashlib.sha256(f.read()).hexdigest()

df_raw = pd.read_csv(DATASET_PATH)
print('=== INFORMASI LINGKUNGAN & AUDIT INTEGRITAS DATASET ===')
print(f'File Dataset   : {os.path.abspath(DATASET_PATH)}')
print(f'SHA-256 Hash   : {dataset_sha256}')
print(f'Dimensi Mentah : {df_raw.shape[0]:,} baris x {df_raw.shape[1]} kolom')
print(f'Python Version : {sys.version.split()[0]} | Scikit-Learn: {sklearn.__version__} | PyTorch: {torch.__version__}')

# 3. Penanganan Missing Values Secara Bersih & Berbasis Run
# Audit nilai missing pada kolom fitur kandidat:
candidate_cols = [
    'ran_prb_cell1', 'ran_snr_cell1',
    'nwdaf_comm_ul_bytes', 'nwdaf_comm_vol_std_bytes',
    'upf_backlog_bytes_embb', 'upf_drops_embb', 'estimated_load_embb'
]
missing_counts = df_raw[candidate_cols].isnull().sum()
print('\nMissing values awal pada fitur kandidat:')
for c in candidate_cols:
    print(f' - {c:26s}: {missing_counts[c]} missing baris')

# Baris sebelum poll pertama NWDAF selesai di awal run dikeluarkan secara transparan
initial_missing_mask = df_raw[candidate_cols].isnull().any(axis=1)
n_dropped = int(initial_missing_mask.sum())
df = df_raw[~initial_missing_mask].copy().reset_index(drop=True)
print(f'\nJumlah baris warm-up awal dikeluarkan: {n_dropped} baris')
print(f'Sampel bersih untuk benchmark konsisten: {len(df):,} baris')

# 4. Pemetaan Kelas Target Ground-Truth
LABEL_MAP = {'BASELINE': 0, 'RAN': 1, 'CORE': 2, 'JOINT': 3}
CLASS_NAMES = ['BASELINE', 'RAN', 'CORE', 'JOINT']
df['target_class'] = df['event_domain'].map(LABEL_MAP)

class_counts = df['target_class'].value_counts().sort_index()
df_target_dist = pd.DataFrame({
    'Class ID': class_counts.index,
    'Domain Label': [CLASS_NAMES[i] for i in class_counts.index],
    'Jumlah Sampel': class_counts.values,
    'Proporsi (%)': (class_counts.values / len(df) * 100.0).round(2)
})
display(df_target_dist)


## Tahap 2: Definisi Protokol Empat Set Fitur Ablasi (Sets A, B, C, D)
- **Tujuan:** Menetapkan empat konfigurasi vektor fitur masukan secara formal untuk studi ablasi lintas domain:
  1. **Set A (RAN-only):** Hanya telemetri antarmuka radio gNB/DU (`ran_prb_cell1`, `ran_snr_cell1`). Domain Core ditiadakan (*ablated*).
  2. **Set B (NWDAF-only):** Hanya analitik native Core 3GPP NWDAF (`nwdaf_comm_ul_bytes`, `nwdaf_comm_vol_std_bytes`). Domain Radio ditiadakan (*ablated*).
  3. **Set C (Cross-Domain Fused — Eksperimen Utama Tesis):** Menggabungkan telemetri radio dan Core NWDAF secara simultan (`ran_prb_cell1`, `ran_snr_cell1`, `nwdaf_comm_ul_bytes`, `nwdaf_comm_vol_std_bytes`).
  4. **Set D (RAN + Kernel OAM — Referensi Batas Atas):** Menggabungkan telemetri radio dengan sensor antrean kernel Linux UPF (`ran_prb_cell1`, `ran_snr_cell1`, `upf_backlog_bytes_embb`, `upf_drops_embb`, `estimated_load_embb`).
> **Catatan Metodologi NWDAF:** Durasi mentah (`nwdaf_comm_duration_sec`) secara ketat dikeluarkan dari fitur prediktor untuk mencegah model belajar umur kumulatif sesi PDU (*session-age chronological leakage*). Kolom volume NWDAF dievaluasi secara eksploratif apa adanya tanpa membagi nilai dengan tiga atau mengubahnya menjadi throughput fisik.


In [ ]:
# Definisi Formal Empat Set Fitur Masukan (Ablasi Simetris)
FEATURE_SETS = {
    'Set A (RAN-only)': [
        'ran_prb_cell1', 'ran_snr_cell1'
    ],
    'Set B (NWDAF-only)': [
        'nwdaf_comm_ul_bytes', 'nwdaf_comm_vol_std_bytes'
    ],
    'Set C (Cross-Domain Fused)': [
        'ran_prb_cell1', 'ran_snr_cell1',
        'nwdaf_comm_ul_bytes', 'nwdaf_comm_vol_std_bytes'
    ],
    'Set D (Kernel OAM Ref)': [
        'ran_prb_cell1', 'ran_snr_cell1',
        'upf_backlog_bytes_embb', 'upf_drops_embb', 'estimated_load_embb'
    ]
}

for s_name, s_cols in FEATURE_SETS.items():
    print(f'{s_name:28s} ({len(s_cols)} fitur): {s_cols}')


## Tahap 3: Partisi Temporal Dalam-Run Bebas Bocoran & Penyelarasan Sekuensial
- **Protokol Partisi Temporal Dalam-Run (*Intra-Run Temporal Split*):**
  - Setiap run independen 600 detik dibagi secara kronologis:
    - **Training (0–60% waktu run):** Digunakan eksklusif untuk fitting model dan penaksiran parameter normalisasi `StandardScaler`.
    - **Validation (60–80% waktu run):** Digunakan untuk monitoring konvergensi dan seleksi model objektif.
    - **Testing (80–100% waktu run):** Disimpan sepenuhnya terisolasi untuk evaluasi performa akhir.
- **Penyelarasan Target Evaluasi Tabular vs GRU:**
  - GRU memerlukan jendela historis $T_{obs} = 10$ detik. Sequence dibentuk **secara terpisah untuk masing-masing run dan partisi**, tanpa pernah melintasi batas partisi atau batas run.
  - Untuk memastikan perbandingan yang adil (*apple-to-apple*), model tabular (M0, M1, M2, M3) dievaluasi pada **titik waktu target yang 100% identik** dengan titik target GRU (sampel ke-10 hingga akhir setiap partisi).


In [ ]:
SEQ_LEN = 10

train_seq_tuples, val_seq_tuples, test_seq_tuples = [], [], []

# Iterasi per run eksperimen untuk menjamin isolasi total
for run_id, grp in df.groupby('run_id', sort=False):
    indices = grp.index.tolist()
    n_total = len(indices)
    
    n_tr = int(n_total * 0.60)
    n_va = int(n_total * 0.20)
    
    tr_indices = indices[:n_tr]
    va_indices = indices[n_tr:n_tr + n_va]
    te_indices = indices[n_tr + n_va:]
    
    # Bentuk sequence HANYA di dalam partisi masing-masing
    for i in range(len(tr_indices) - SEQ_LEN + 1):
        train_seq_tuples.append((tr_indices[i:i + SEQ_LEN], tr_indices[i + SEQ_LEN - 1]))
    for i in range(len(va_indices) - SEQ_LEN + 1):
        val_seq_tuples.append((va_indices[i:i + SEQ_LEN], va_indices[i + SEQ_LEN - 1]))
    for i in range(len(te_indices) - SEQ_LEN + 1):
        test_seq_tuples.append((te_indices[i:i + SEQ_LEN], te_indices[i + SEQ_LEN - 1]))

# Ekstraksi indeks target yang diselaraskan
tr_targets = np.array([x[1] for x in train_seq_tuples])
va_targets = np.array([x[1] for x in val_seq_tuples])
te_targets = np.array([x[1] for x in test_seq_tuples])

y_train_aligned = df.loc[tr_targets, 'target_class'].values
y_val_aligned = df.loc[va_targets, 'target_class'].values
y_test_aligned = df.loc[te_targets, 'target_class'].values

print('=== HASIL PENYELARASAN TITIK TARGET EVALUASI TABULAR & GRU ===')
print(f'Train Target Points : {len(tr_targets):,} sampel')
print(f'Val Target Points   : {len(va_targets):,} sampel')
print(f'Test Target Points  : {len(te_targets):,} sampel')

# Tampilkan distribusi target pada data uji independen
df_te_dist = pd.DataFrame({
    'Class ID': range(4),
    'Label': CLASS_NAMES,
    'Jumlah Test Target': [(y_test_aligned == c).sum() for c in range(4)],
    'Prevalensi Dasar P_c': [round((y_test_aligned == c).mean(), 4) for c in range(4)]
})
display(df_te_dist)

# 2. Fitting StandardScaler Eksklusif pada Training Set
SCALERS = {}
TABULAR_DATA = {}
SEQUENCE_DATA = {}

for s_name, cols in FEATURE_SETS.items():
    scaler = StandardScaler()
    
    # Fit HANYA pada data baris pelatihan
    all_tr_rows = []
    for run_id, grp in df.groupby('run_id', sort=False):
        n_tr = int(len(grp) * 0.60)
        all_tr_rows.extend(grp.index[:n_tr].tolist())
    
    scaler.fit(df.loc[all_tr_rows, cols].values)
    SCALERS[s_name] = scaler
    
    # Transformasi fitur untuk tabular (pada target yang diselaraskan)
    X_tr_tab = scaler.transform(df.loc[tr_targets, cols].values)
    X_va_tab = scaler.transform(df.loc[va_targets, cols].values)
    X_te_tab = scaler.transform(df.loc[te_targets, cols].values)
    
    TABULAR_DATA[s_name] = {
        'X_train': X_tr_tab,
        'X_val': X_va_tab,
        'X_test': X_te_tab
    }
    
    # Bentuk sequence 3D (N, SEQ_LEN, D) menggunakan data yang sudah ditransformasi
    df_scaled_full = pd.DataFrame(scaler.transform(df[cols].values), index=df.index)
    
    def build_seq_array(tuples_list):
        arr = np.zeros((len(tuples_list), SEQ_LEN, len(cols)), dtype=np.float32)
        for idx, (seq_idxs, tgt_idx) in enumerate(tuples_list):
            arr[idx] = df_scaled_full.loc[seq_idxs].values
        return arr
    
    SEQUENCE_DATA[s_name] = {
        'X_train_seq': build_seq_array(train_seq_tuples),
        'X_val_seq': build_seq_array(val_seq_tuples),
        'X_test_seq': build_seq_array(test_seq_tuples)
    }

print('StandardScaler & tensor sekuensial berhasil difit secara terisolasi.')


## Tahap 4: Pelatihan Model, Protokol Pengukuran Latensi & Perhitungan Metrik Lengkap
- **Model Benchmark:**
  - `M0: Heuristic Rule-Based` (Thresholding fisik diskrit)
  - `M1: Decision Tree (CART)` (`max_depth=5`, `class_weight='balanced'`)
  - `M2: Random Forest` (`n_estimators=100`, `max_depth=8`, `class_weight='balanced'`)
  - `M3: Multi-Layer Perceptron (MLP)` (`hidden_layer_sizes=(64, 32)`, `max_iter=300`)
  - `M4: Recurrent GRU` (`hidden_dim=32`, `num_layers=1`, `seq_len=10s`)
- **Protokol Latensi:**
  - Waktu diukur per sampel tunggal (*batch size = 1*) berulang sebanyak **200 iterasi setelah 50 iterasi pemanasan (*warm-up*)**.
  - Nilai yang dilaporkan adalah **Median** dan **p95** dari distribusi pengukuran nyata pada CPU (termasuk scaling).
- **Protokol AUPRC & Average Precision (AP):**
  - Dihitung secara *One-vs-Rest* menggunakan probabilitas kontinu (`predict_proba` atau softmax).
  - `average_precision_score` untuk AP, dan integrasi trapezoidal `auc(recall, precision)` untuk AUPRC.
  - Model M0 yang menghasilkan keputusan diskrit ditandai `N/A` pada AP/AUPRC karena tidak memiliki skor kontinu.


In [ ]:
# 1. Fungsi Heuristic Rule-Based M0
def predict_heuristic_labels(X_raw, set_name):
    preds = []
    for row in X_raw:
        if 'RAN-only' in set_name:
            prb = row[0]
            preds.append(1 if prb > 85.0 else 0)
        elif 'NWDAF-only' in set_name:
            vol_ul = row[0]
            vol_std = row[1]
            if vol_std < 100000.0 and vol_std > 0.0:
                preds.append(2)  # CORE
            elif vol_std > 900000.0:
                preds.append(1)  # RAN
            elif vol_std > 500000.0 and vol_ul > 1.5e9:
                preds.append(3)  # JOINT
            else:
                preds.append(0)  # BASELINE
        elif 'Cross-Domain Fused' in set_name:
            prb = row[0]
            vol_std = row[3]
            if prb > 85.0 and vol_std < 800000.0:
                preds.append(3)  # JOINT
            elif prb > 85.0:
                preds.append(1)  # RAN
            elif vol_std < 100000.0 and vol_std > 0.0:
                preds.append(2)  # CORE
            else:
                preds.append(0)  # BASELINE
        elif 'Kernel OAM' in set_name:
            prb = row[0]
            backlog = row[2]
            if prb > 85.0 and backlog > 50000.0:
                preds.append(3)  # JOINT
            elif prb > 85.0:
                preds.append(1)  # RAN
            elif backlog > 50000.0:
                preds.append(2)  # CORE
            else:
                preds.append(0)  # BASELINE
    return np.array(preds)

# 2. Arsitektur GRU PyTorch
class GRUModel(nn.Module):
    def __init__(self, in_features, hidden_dim=32, num_classes=4):
        super().__init__()
        self.gru = nn.GRU(in_features, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

# 3. Benchmark Pengukuran Latensi Inferensi Berulang
def measure_inference_latency(predict_fn, sample_input, warmup=50, reps=200):
    for _ in range(warmup):
        predict_fn(sample_input)
    latencies_ms = []
    for _ in range(reps):
        t0 = time.perf_counter()
        predict_fn(sample_input)
        t1 = time.perf_counter()
        latencies_ms.append((t1 - t0) * 1000.0)
    return float(np.median(latencies_ms)), float(np.percentile(latencies_ms, 95))

# 4. Container Penyimpanan Metrik & Kurva
benchmark_summary = []
per_class_summary = []
CONF_MATRICES = {}
PR_CURVES = {}
PREDICTIONS_DICT = {}

MODELS_LIST = ['M0: Heuristic', 'M1: Decision Tree', 'M2: Random Forest', 'M3: MLP', 'M4: Recurrent GRU']

for set_name, cols in FEATURE_SETS.items():
    print(f'Mengevaluasi {set_name} ...')
    X_tr = TABULAR_DATA[set_name]['X_train']
    X_va = TABULAR_DATA[set_name]['X_val']
    X_te = TABULAR_DATA[set_name]['X_test']
    
    X_tr_raw = df.loc[tr_targets, cols].values
    X_va_raw = df.loc[va_targets, cols].values
    X_te_raw = df.loc[te_targets, cols].values
    
    X_tr_seq = SEQUENCE_DATA[set_name]['X_train_seq']
    X_va_seq = SEQUENCE_DATA[set_name]['X_val_seq']
    X_te_seq = SEQUENCE_DATA[set_name]['X_test_seq']
    
    # Evaluasi setiap model
    for m_name in MODELS_LIST:
        y_prob_te = None
        
        if m_name == 'M0: Heuristic':
            y_pred_va = predict_heuristic_labels(X_va_raw, set_name)
            y_pred_te = predict_heuristic_labels(X_te_raw, set_name)
            lat_med, lat_p95 = measure_inference_latency(lambda x: predict_heuristic_labels(x, set_name), X_te_raw[0:1])
            
        elif m_name == 'M1: Decision Tree':
            model = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=RANDOM_SEED)
            model.fit(X_tr, y_train_aligned)
            y_pred_va = model.predict(X_va)
            y_pred_te = model.predict(X_te)
            y_prob_te = model.predict_proba(X_te)
            lat_med, lat_p95 = measure_inference_latency(model.predict_proba, X_te[0:1])
            
        elif m_name == 'M2: Random Forest':
            model = RandomForestClassifier(n_estimators=100, max_depth=8, class_weight='balanced', random_state=RANDOM_SEED)
            model.fit(X_tr, y_train_aligned)
            y_pred_va = model.predict(X_va)
            y_pred_te = model.predict(X_te)
            y_prob_te = model.predict_proba(X_te)
            lat_med, lat_p95 = measure_inference_latency(model.predict_proba, X_te[0:1])
            
        elif m_name == 'M3: MLP':
            model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=RANDOM_SEED)
            model.fit(X_tr, y_train_aligned)
            y_pred_va = model.predict(X_va)
            y_pred_te = model.predict(X_te)
            y_prob_te = model.predict_proba(X_te)
            lat_med, lat_p95 = measure_inference_latency(model.predict_proba, X_te[0:1])
            
        elif m_name == 'M4: Recurrent GRU':
            gru_net = GRUModel(len(cols), hidden_dim=32, num_classes=4)
            opt = torch.optim.Adam(gru_net.parameters(), lr=0.01)
            crit = nn.CrossEntropyLoss()
            
            X_tr_t = torch.tensor(X_tr_seq, dtype=torch.float32)
            y_tr_t = torch.tensor(y_train_aligned, dtype=torch.long)
            
            gru_net.train()
            for ep in range(25):
                opt.zero_grad()
                loss = crit(gru_net(X_tr_t), y_tr_t)
                loss.backward()
                opt.step()
            
            gru_net.eval()
            with torch.no_grad():
                logits_va = gru_net(torch.tensor(X_va_seq, dtype=torch.float32))
                logits_te = gru_net(torch.tensor(X_te_seq, dtype=torch.float32))
                y_pred_va = logits_va.argmax(dim=1).numpy()
                y_pred_te = logits_te.argmax(dim=1).numpy()
                y_prob_te = torch.softmax(logits_te, dim=1).numpy()
            
            sample_seq_t = torch.tensor(X_te_seq[0:1], dtype=torch.float32)
            def run_gru(x):
                with torch.no_grad():
                    return gru_net(x)
            lat_med, lat_p95 = measure_inference_latency(run_gru, sample_seq_t)
        
        # Perhitungan Metrik Global & Macro
        val_macro_f1 = f1_score(y_val_aligned, y_pred_va, average='macro', zero_division=0)
        test_macro_f1 = f1_score(y_test_aligned, y_pred_te, average='macro', zero_division=0)
        test_weighted_f1 = f1_score(y_test_aligned, y_pred_te, average='weighted', zero_division=0)
        test_macro_p = precision_score(y_test_aligned, y_pred_te, average='macro', zero_division=0)
        test_macro_r = recall_score(y_test_aligned, y_pred_te, average='macro', zero_division=0)
        test_acc = accuracy_score(y_test_aligned, y_pred_te)
        test_bal_acc = balanced_accuracy_score(y_test_aligned, y_pred_te)
        
        # Perhitungan Metrik Per-Kelas & AUPRC / AP
        p_c, r_c, f1_c, supp_c = precision_recall_fscore_support(y_test_aligned, y_pred_te, labels=[0, 1, 2, 3], zero_division=0)
        
        ap_per_class = []
        auprc_per_class = []
        
        for c in range(4):
            y_true_binary = (y_test_aligned == c).astype(int)
            
            if y_prob_te is not None:
                scores_c = y_prob_te[:, c]
                ap_val = average_precision_score(y_true_binary, scores_c)
                prec_curve, rec_curve, _ = precision_recall_curve(y_true_binary, scores_c)
                auprc_val = auc(rec_curve, prec_curve)
                ap_per_class.append(ap_val)
                auprc_per_class.append(auprc_val)
                PR_CURVES[(set_name, m_name, c)] = (rec_curve, prec_curve, ap_val, auprc_val)
            else:
                ap_per_class.append(np.nan)
                auprc_per_class.append(np.nan)
                PR_CURVES[(set_name, m_name, c)] = (r_c[c], p_c[c], np.nan, np.nan)
            
            per_class_summary.append({
                'Set Fitur': set_name,
                'Model': m_name,
                'Class ID': c,
                'Class Label': CLASS_NAMES[c],
                'Precision': round(p_c[c], 4),
                'Recall': round(r_c[c], 4),
                'F1-score': round(f1_c[c], 4),
                'Support': int(supp_c[c]),
                'AP': round(ap_per_class[-1], 4) if not np.isnan(ap_per_class[-1]) else np.nan,
                'AUPRC': round(auprc_per_class[-1], 4) if not np.isnan(auprc_per_class[-1]) else np.nan
            })
        
        macro_ap = float(np.nanmean(ap_per_class)) if y_prob_te is not None else np.nan
        macro_auprc = float(np.nanmean(auprc_per_class)) if y_prob_te is not None else np.nan
        
        # Confusion Matrix Raw & Normalized
        cm_raw = confusion_matrix(y_test_aligned, y_pred_te, labels=[0, 1, 2, 3])
        cm_norm = confusion_matrix(y_test_aligned, y_pred_te, labels=[0, 1, 2, 3], normalize='true')
        CONF_MATRICES[(set_name, m_name)] = {'raw': cm_raw, 'norm': cm_norm}
        
        benchmark_summary.append({
            'Set Fitur': set_name,
            'Model': m_name,
            'Val Macro-F1': round(val_macro_f1, 4),
            'Test Macro-F1': round(test_macro_f1, 4),
            'Test Weighted-F1': round(test_weighted_f1, 4),
            'Test Macro-P': round(test_macro_p, 4),
            'Test Macro-R': round(test_macro_r, 4),
            'Accuracy': round(test_acc, 4),
            'Balanced Acc': round(test_bal_acc, 4),
            'Macro-AP': round(macro_ap, 4) if not np.isnan(macro_ap) else np.nan,
            'Macro-AUPRC': round(macro_auprc, 4) if not np.isnan(macro_auprc) else np.nan,
            'Lat Median (ms)': round(lat_med, 4),
            'Lat p95 (ms)': round(lat_p95, 4)
        })

df_benchmark = pd.DataFrame(benchmark_summary)
df_per_class = pd.DataFrame(per_class_summary)
print('Benchmark 20 konfigurasi berhasil diselesaikan.')


## Tahap 5: Penyajian Hasil Komputasi & Visualisasi Terkelompok
- **Tujuan:** Menampilkan seluruh hasil evaluasi secara visual dan tabel data yang dihasilkan langsung dari eksekusi tanpa teks statis interpretatif.
- **Visualisasi Terkelompok:**
  1. **Tabel Ringkasan Komparatif Lengkap:** 20 konfigurasi (5 model × 4 set fitur).
  2. **Heatmap Ringkasan Model × Set Fitur:** 4 metrik macro (Precision, Recall, F1, AUPRC).
  3. **Heatmap Performa Per Kelas:** Precision, Recall, F1 untuk BASELINE, RAN, CORE, JOINT.
  4. **Confusion Matrix (Counts & Normalized):** Ditampilkan untuk model terbaik yang dipilih berdasarkan kriteria objektif Validation Macro-F1 pada Set C.
  5. **Kurva Precision–Recall per Kelas:** Dengan baseline prevalensi kelas, anotasi AP dan AUPRC trapezoidal, serta titik operasi M0.
  6. **Perbandingan Validation vs Test:** Mengidentifikasi generalisasi dan gap performa antar-partisi.
  7. **Grafik Delta Gain Set C:** Kenaikan performa Set C terhadap Set A dan B yang dihitung secara dinamis.
  8. **Frontier Pareto Akurasi vs Latensi:** Scatter plot Test Macro-F1 terhadap Latensi p95.


In [ ]:
# 1. Tabel Ringkasan Komparatif Lengkap (20 Konfigurasi)
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)
print('=== TABEL HASIL BENCHMARK MODEL & ABLASI FITUR 4-TIER ===')
display(df_benchmark.sort_values(by=['Model', 'Set Fitur']).reset_index(drop=True))


In [ ]:
# 2. Heatmap Ringkasan Metrik Macro (4 Panel)
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

macro_metrics = [
    ('Test Macro-P', 'Macro Precision', axes[0, 0]),
    ('Test Macro-R', 'Macro Recall', axes[0, 1]),
    ('Test Macro-F1', 'Macro F1-Score', axes[1, 0]),
    ('Macro-AUPRC', 'Macro AUPRC (OvR)', axes[1, 1])
]

for col_name, title, ax in macro_metrics:
    pivot_df = df_benchmark.pivot(index='Model', columns='Set Fitur', values=col_name)
    sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0.0, vmax=1.0,
                linewidths=0.5, ax=ax, cbar_kws={'label': col_name})
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.suptitle('Ringkasan Metrik Macro Model × Set Fitur pada Test Set', fontsize=12.5, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# 3. Heatmap F1-Score Per-Kelas untuk BASELINE, RAN, CORE, JOINT (4 Panel)
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for c in range(4):
    c_name = CLASS_NAMES[c]
    sub_df = df_per_class[df_per_class['Class Label'] == c_name]
    pivot_c = sub_df.pivot(index='Model', columns='Set Fitur', values='F1-score')
    sns.heatmap(pivot_c, annot=True, fmt='.3f', cmap='Blues', vmin=0.0, vmax=1.0,
                linewidths=0.5, ax=axes[c], cbar=False)
    axes[c].set_title(f'F1-Score: {c_name}', fontweight='bold', fontsize=11)
    axes[c].set_xlabel('')
    axes[c].set_ylabel('Model' if c == 0 else '')

plt.suptitle('Perbandingan F1-Score Per-Kelas Lintas Model & Set Fitur', fontsize=12.5, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 4. Seleksi Model Terbaik Berdasarkan Validation Set (Kriteria Objektif: Max Val Macro-F1 pada Set C)
val_set_c = df_benchmark[df_benchmark['Set Fitur'] == 'Set C (Cross-Domain Fused)']
best_row = val_set_c.sort_values(by='Val Macro-F1', ascending=False).iloc[0]
selected_model = best_row['Model']
print(f'Model terpilih berdasarkan Validation Macro-F1 tertinggi pada Set C: {selected_model} (Val F1: {best_row["Val Macro-F1"]:.4f})')

# Visualisasi Matriks Konfusi Raw (Baris 1) dan Ternormalisasi (Baris 2)
fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))
sets_ordered = list(FEATURE_SETS.keys())

for j, s_name in enumerate(sets_ordered):
    cm_data = CONF_MATRICES[(s_name, selected_model)]
    
    # Row 1: Raw Counts
    sns.heatmap(cm_data['raw'], annot=True, fmt='d', cmap='Blues', ax=axes[0, j],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES if j == 0 else False, cbar=False)
    axes[0, j].set_title(f'{s_name}\n[Raw Counts]', fontweight='bold', fontsize=10)
    axes[0, j].set_ylabel('True Label' if j == 0 else '')
    
    # Row 2: Normalized by True Class
    sns.heatmap(cm_data['norm'], annot=True, fmt='.2f', cmap='Greens', ax=axes[1, j],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES if j == 0 else False, vmin=0.0, vmax=1.0, cbar=False)
    axes[1, j].set_title(f'{s_name}\n[Normalized Recall]', fontweight='bold', fontsize=10)
    axes[1, j].set_xlabel('Predicted Label')
    axes[1, j].set_ylabel('True Label' if j == 0 else '')

plt.suptitle(f'Matriks Konfusi {selected_model} pada Test Set (Raw Counts & Normalized)', fontsize=12.5, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


In [ ]:
# 5. Kurva Precision-Recall (One-vs-Rest) per Kelas dengan Prevalensi Baseline
fig, axes = plt.subplots(1, 4, figsize=(19, 4.5))
colors_sets = {'Set A (RAN-only)': '#7f7f7f', 'Set B (NWDAF-only)': '#ff7f0e', 'Set C (Cross-Domain Fused)': '#1f77b4', 'Set D (Kernel OAM Ref)': '#2ca02c'}

for c in range(4):
    c_name = CLASS_NAMES[c]
    prevalence = (y_test_aligned == c).mean()
    ax = axes[c]
    
    # Garis prevalensi acak baseline
    ax.axhline(prevalence, color='black', ls=':', lw=1.2, label=f'Prevalensi Dasar ({prevalence:.2f})')
    
    for s_name in sets_ordered:
        curve_data = PR_CURVES[(s_name, selected_model, c)]
        rec, prec, ap_val, auprc_val = curve_data
        
        if not np.isnan(ap_val):
            lbl = f'{s_name.split()[1]}: AP={ap_val:.2f}, AUPRC={auprc_val:.2f}'
            ax.plot(rec, prec, lw=1.4, label=lbl, color=colors_sets[s_name])
        else:
            ax.scatter(rec, prec, color=colors_sets[s_name], s=50, zorder=5, label=f'{s_name.split()[1]} (titik)')
    
    # Tambahkan titik operasi M0 Heuristic
    h_rec, h_prec, _, _ = PR_CURVES[(sets_ordered[2], 'M0: Heuristic', c)]
    ax.scatter(h_rec, h_prec, color='crimson', marker='X', s=70, zorder=6, label=f'M0 Heuristic (Set C)')
    
    ax.set_title(f'Kelas: {c_name} (OvR)', fontweight='bold', fontsize=11)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision' if c == 0 else '')
    ax.set_xlim(-0.02, 1.05)
    ax.set_ylim(-0.02, 1.08)
    ax.grid(True, ls='--', alpha=0.5)
    ax.legend(loc='lower left', fontsize=7.5, frameon=True)

plt.suptitle(f'Kurva Precision-Recall One-vs-Rest ({selected_model}) Lintas Set Fitur', fontsize=12.5, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 6. Perbandingan Validation vs Test Macro-F1 (Evaluasi Drop / Generalisasi)
fig, ax = plt.subplots(figsize=(13, 5))

models = MODELS_LIST
x = np.arange(len(models))
width = 0.09

for i, s_name in enumerate(sets_ordered):
    sub_s = df_benchmark[df_benchmark['Set Fitur'] == s_name]
    val_f1 = [sub_s[sub_s['Model'] == m]['Val Macro-F1'].values[0] for m in models]
    test_f1 = [sub_s[sub_s['Model'] == m]['Test Macro-F1'].values[0] for m in models]
    
    pos_val = x + (i * 2 - 3.5) * width
    pos_test = x + (i * 2 - 2.5) * width
    
    ax.bar(pos_val, val_f1, width, label=f'{s_name.split()[1]} [Val]', alpha=0.5, color=colors_sets[s_name], edgecolor='black')
    ax.bar(pos_test, test_f1, width, label=f'{s_name.split()[1]} [Test]', alpha=0.9, color=colors_sets[s_name])

ax.set_ylabel('Macro-F1 Score', fontweight='bold')
ax.set_title('Perbandingan Macro-F1: Validation Set (Transparan) vs Test Set (Solid)', fontweight='bold', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels([m.replace(': ', '\n') for m in models], fontsize=9)
ax.set_ylim(0, 1.15)
ax.grid(True, ls='--', alpha=0.5)
ax.legend(loc='upper left', ncol=4, fontsize=7.5, frameon=True)

plt.tight_layout()
plt.show()

# 7. Grafik Delta Gain Set C terhadap Set A dan Set B (Dihitung dari Hasil Aktual)
fig, ax = plt.subplots(figsize=(10, 4.5))

delta_vs_a = []
delta_vs_b = []

for m in models:
    f1_a = df_benchmark[(df_benchmark['Model'] == m) & (df_benchmark['Set Fitur'] == sets_ordered[0])]['Test Macro-F1'].values[0]
    f1_b = df_benchmark[(df_benchmark['Model'] == m) & (df_benchmark['Set Fitur'] == sets_ordered[1])]['Test Macro-F1'].values[0]
    f1_c = df_benchmark[(df_benchmark['Model'] == m) & (df_benchmark['Set Fitur'] == sets_ordered[2])]['Test Macro-F1'].values[0]
    delta_vs_a.append(f1_c - f1_a)
    delta_vs_b.append(f1_c - f1_b)

bar_w = 0.35
r1 = ax.bar(x - bar_w/2, delta_vs_a, bar_w, label='Delta Gain: Set C (Fused) - Set A (RAN-only)', color='steelblue', alpha=0.9)
r2 = ax.bar(x + bar_w/2, delta_vs_b, bar_w, label='Delta Gain: Set C (Fused) - Set B (NWDAF-only)', color='coral', alpha=0.9)

ax.axhline(0, color='black', lw=1.0)
ax.set_ylabel('Perubahan Macro-F1 (Delta Gain)', fontweight='bold')
ax.set_title('Delta Peningkatan Akurasi Set C (Cross-Domain) terhadap Single-Domain Baseline', fontweight='bold', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels([m.replace(': ', '\n') for m in models], fontsize=9)
ax.grid(True, ls='--', alpha=0.5)
ax.legend(loc='upper right', frameon=True)

for rects in [r1, r2]:
    for rect in rects:
        h = rect.get_height()
        ax.annotate(f'{h:+.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3 if h >= 0 else -10), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# 8. Frontier Pareto: Akurasi Diagnostik vs Latensi Inferensi Pengukuran Nyata
fig, ax = plt.subplots(figsize=(10.5, 5.5))

markers_model = {
    'M0: Heuristic': 'o',
    'M1: Decision Tree': 's',
    'M2: Random Forest': '^',
    'M3: MLP': 'D',
    'M4: Recurrent GRU': 'P'
}

for idx, row in df_benchmark.iterrows():
    s_name = row['Set Fitur']
    m_name = row['Model']
    lat_p95 = row['Lat p95 (ms)']
    f1 = row['Test Macro-F1']
    
    ax.scatter(lat_p95, f1, color=colors_sets[s_name], marker=markers_model[m_name], s=120, alpha=0.85,
               edgecolor='black', linewidth=0.6)
    ax.annotate(f'{m_name.split(": ")[0]}', xy=(lat_p95, f1), xytext=(4, 2), textcoords='offset points', fontsize=7.5)

# Garis Batas Rekayasa Non-RT RIC rApp Testbed (<10 ms)
ax.axvline(10.0, color='red', ls=':', lw=1.2, label='Target Rekayasa Non-RT RIC rApp Testbed (10 ms)')

ax.set_xlabel('Inference Latency p95 per sampel tunggal (ms) - Skala Log', fontweight='bold')
ax.set_ylabel('Test Macro-F1 Score', fontweight='bold')
ax.set_xscale('log')
ax.set_ylim(-0.05, 1.12)
ax.set_title('Frontier Pareto: Trade-Off Akurasi Diagnostik vs Latensi Inferensi (Pengukuran Nyata CPU, Batch=1)', fontweight='bold', fontsize=11)
ax.grid(True, ls='--', alpha=0.5)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Set A: RAN-only', markerfacecolor=colors_sets[sets_ordered[0]], markersize=8),
    Line2D([0], [0], marker='o', color='w', label='Set B: NWDAF-only', markerfacecolor=colors_sets[sets_ordered[1]], markersize=8),
    Line2D([0], [0], marker='o', color='w', label='Set C: Cross-Domain Fused', markerfacecolor=colors_sets[sets_ordered[2]], markersize=8),
    Line2D([0], [0], marker='o', color='w', label='Set D: Kernel OAM Ref', markerfacecolor=colors_sets[sets_ordered[3]], markersize=8),
    Line2D([0], [0], color='red', ls=':', lw=1.2, label='Target Rekayasa Testbed (10 ms)')
]
ax.legend(handles=legend_elements, loc='lower right', frameon=True, fontsize=8)

plt.tight_layout()
plt.show()
